# 第19章 分类

本 notebook 用一个小型 SDSS 风格教学数据集说明分类任务：建立规则 baseline，构造最小 kNN 分类器，再用混淆矩阵查看错误分布。

## 学习目标

- 理解分类任务输出的是离散标签。
- 对比规则 baseline 和数据驱动分类器。
- 初步读懂混淆矩阵与按类别召回率。
- 认识到小样本高分只能说明教学流程跑通，不代表真实科学可用。

In [ ]:
import csv
import math
from pathlib import Path

data_path = Path("../../data/small/object_classification_demo.csv")
with data_path.open(newline="", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

for row in rows:
    for key in ("u_g", "g_r", "r_i", "i_z"):
        row[key] = float(row[key])

print("loaded rows:", len(rows))
print("classes:", sorted({row['object_class'] for row in rows}))


## 一个固定的教学切分

为了让每一类在训练集和测试集中都出现，我们用固定索引做一个教学版切分。

In [ ]:
test_indices = {1, 4, 5, 8, 11, 14}
train_rows = [row for index, row in enumerate(rows) if index not in test_indices]
test_rows = [row for index, row in enumerate(rows) if index in test_indices]

print("train:", len(train_rows), "test:", len(test_rows))
print("test objects:", [row["object_id"] for row in test_rows])


## 规则 baseline

先写一个很简单的颜色阈值分类器。它不是最优模型，但它给我们提供了一个可解释的比较起点。

In [ ]:
def classify_rule(row):
    if row["u_g"] < 0.5 and row["g_r"] < 0.1:
        return "quasar"
    if row["g_r"] > 0.7 and row["r_i"] > 0.3:
        return "galaxy"
    return "star"


rule_predictions = [classify_rule(row) for row in test_rows]
list(zip([row["object_id"] for row in test_rows], [row["object_class"] for row in test_rows], rule_predictions))


## 一个最小 kNN 分类器

这里使用三个颜色特征，构造一个不依赖第三方机器学习库的 kNN 实现。

In [ ]:
def feature_vector(row):
    return (row["u_g"], row["g_r"], row["r_i"])


def euclidean_distance(a_value, b_value):
    return math.sqrt(sum((ax - bx) ** 2 for ax, bx in zip(a_value, b_value)))


def classify_knn(target_row, training_rows, k_neighbors=3):
    distances = []
    target_features = feature_vector(target_row)
    for row in training_rows:
        distances.append((euclidean_distance(target_features, feature_vector(row)), row["object_class"]))
    distances.sort(key=lambda item: item[0])

    votes = {}
    for _, label in distances[:k_neighbors]:
        votes[label] = votes.get(label, 0) + 1

    return max(sorted(votes), key=lambda label: votes[label])


knn_predictions = [classify_knn(row, train_rows, k_neighbors=3) for row in test_rows]
list(zip([row["object_id"] for row in test_rows], [row["object_class"] for row in test_rows], knn_predictions))


## 指标：准确率、混淆矩阵、按类别召回率

分类任务里，单看准确率不够。我们还需要知道模型具体把哪些类别混淆了。

In [ ]:
labels = sorted({row["object_class"] for row in rows})
truths = [row["object_class"] for row in test_rows]


def accuracy_score(y_true, y_pred):
    return sum(1 for truth, pred in zip(y_true, y_pred) if truth == pred) / len(y_true)


def confusion_matrix(y_true, y_pred, labels):
    matrix = {truth: {pred: 0 for pred in labels} for truth in labels}
    for truth, pred in zip(y_true, y_pred):
        matrix[truth][pred] += 1
    return matrix


def per_class_recall(matrix, labels):
    recalls = {}
    for label in labels:
        total = sum(matrix[label].values())
        recalls[label] = matrix[label][label] / total if total else 0.0
    return recalls


rule_accuracy = accuracy_score(truths, rule_predictions)
knn_accuracy = accuracy_score(truths, knn_predictions)
rule_matrix = confusion_matrix(truths, rule_predictions, labels)
knn_matrix = confusion_matrix(truths, knn_predictions, labels)

print("rule accuracy =", round(rule_accuracy, 3))
print("knn accuracy =", round(knn_accuracy, 3))
print("rule recall =", per_class_recall(rule_matrix, labels))
print("knn recall =", per_class_recall(knn_matrix, labels))


In [ ]:
print("rule confusion matrix =", rule_matrix)
print("knn confusion matrix =", knn_matrix)


## 如果安装了 scikit-learn

下面这个单元是可选的。它把最小教学实现与逻辑回归、决策树这些标准工具连起来。

In [ ]:
try:
    from sklearn.linear_model import LogisticRegression
    from sklearn.tree import DecisionTreeClassifier
except ModuleNotFoundError:
    print("scikit-learn 未安装；已跳过标准库分类示例。")
else:
    x_train = [list(feature_vector(row)) for row in train_rows]
    y_train = [row["object_class"] for row in train_rows]
    x_test = [list(feature_vector(row)) for row in test_rows]

    logistic_model = LogisticRegression(max_iter=1000)
    logistic_model.fit(x_train, y_train)
    logistic_predictions = logistic_model.predict(x_test)
    print("logistic accuracy =", round(accuracy_score(truths, logistic_predictions), 3))

    tree_model = DecisionTreeClassifier(max_depth=3, random_state=42)
    tree_model.fit(x_train, y_train)
    tree_predictions = tree_model.predict(x_test)
    print("decision-tree accuracy =", round(accuracy_score(truths, tree_predictions), 3))


## 小结

这个教学例子说明了分类任务的一个核心节奏：先有可解释的规则 baseline，再引入数据驱动模型，并用混淆矩阵检查错误到底发生在哪里。